# 01 — Qwen-27B Extraction Probe
Goal: before building the pipeline, judge if your llama.cpp Qwen-27B extracts usable Novelty / Method / Limitations / Future Work + Concept Hubs + score.

Run this **on the server** (has GPUs + llama.cpp). No Obisidian / pipeline needed.

In [1]:
import os, json, requests

BASE_URL = os.environ.get("LLAMA_BASE_URL", " https://divorcee-work-pessimism.ngrok-free.dev/v1")
MODEL = os.environ.get("LLAMA_MODEL", "Swift-Qwen3.8-27B-Q4_K_M")
print(f"BASE_URL={BASE_URL} MODEL={MODEL}")

def chat(prompt: str, temp: float = 0.2, max_tokens: int = 1200) -> str:
    r = requests.post(
        f"{BASE_URL}/chat/completions",
        json={"model": MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": temp, "max_tokens": max_tokens},
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

BASE_URL= https://divorcee-work-pessimism.ngrok-free.dev/v1 MODEL=Swift-Qwen3.8-27B-Q4_K_M


## 1. Sample papers (inline so notebook runs offline)
Swap in your own thesis + abstracts next.

In [2]:
THESIS = "We study the application of deep learning methods into areas of In-Vehicle Network"

SAMPLES = [
    # ============================================================
    # DIRECTION 1: Knowledge Distillation for Ultra-Lightweight CAN IDS
    # ============================================================
    {
        "id": "kd_can_ids",
        "title": "Knowledge Distillation for Ultra-Lightweight Intrusion Detection in Controller Area Networks",
        "abstract": (
            "We propose a knowledge distillation framework that transfers knowledge from a large, high-capacity "
            "teacher model (Graph Neural Network or Transformer) to a tiny student model deployable on "
            "resource-constrained ECUs. The teacher is trained on public CAN datasets (Car-Hacking, ROAD, CAN-FD) "
            "and achieves state-of-the-art accuracy. The student uses depthwise separable convolutions and "
            "pruned GNN layers, achieving <10K parameters and <10 KB memory footprint while retaining >99% "
            "of teacher accuracy. We evaluate on ARM Cortex-M4 and show inference latency under 1 ms. "
            "Contribution: First systematic study of knowledge distillation for graph-based CAN IDS; "
            "demonstrates feasibility of on-ECU deployment without accuracy loss. "
            "Limitation: Distillation is evaluated only on closed-set classification; open-set detection "
            "degrades significantly in the student model. Cross-dataset generalization remains untested. "
            "Future work: Combine distillation with open-set learning, test on CAN-FD and automotive Ethernet, "
            "explore quantization-aware distillation, and validate on real ECU hardware."
        ),
    },

    # ============================================================
    # DIRECTION 2: Small Language Models (SLMs) for CAN IDS
    # ============================================================
    {
        "id": "slm_can_ids",
        "title": "Small Language Models for Efficient and Explainable CAN Bus Intrusion Detection",
        "abstract": (
            "We investigate the use of Small Language Models (MiniLM, DistilBERT, TinyBERT) for CAN intrusion "
            "detection by converting CAN traffic into text-like token sequences (ID, payload bytes, timing). "
            "MiniLM achieves the best accuracy-efficiency tradeoff, while TinyBERT is more compact but less "
            "accurate. We benchmark on Car-Hacking and ROAD datasets and compare against GNN and CNN baselines. "
            "Contribution: First evaluation of SLMs for CAN IDS; demonstrates that compact language models can "
            "match CNN performance while providing attention-based interpretability. "
            "Limitation: CAN-to-text tokenization loses precise numerical relationships; SLMs struggle with "
            "rare attack classes and open-set scenarios. Inference on embedded hardware is not yet validated. "
            "Future work: Design CAN-specific tokenizers, integrate few-shot or prototypical learning for rare "
            "attacks, explore knowledge distillation from SLMs to even smaller models, and test on Jetson/ECU."
        ),
    },

    # ============================================================
    # DIRECTION 3: Open-Set and Few-Shot Learning on New Public Datasets
    # ============================================================
    {
        "id": "openset_fewshot_can",
        "title": "Open-Set and Few-Shot Learning for Cross-Dataset CAN Intrusion Detection",
        "abstract": (
            "We study domain shift and open-world challenges in CAN IDS by training on Car-Hacking and testing "
            "on newer, more realistic datasets (CIC-IoV-2024, CAN-ML, ROAD). We extend our OpenSetGNN and "
            "Prototypical Network frameworks with domain adaptation and few-shot fine-tuning. Results show "
            "that models trained on older datasets degrade significantly under domain shift, but few-shot "
            "adaptation with 5-10 samples per class recovers >95% of in-domain performance. "
            "Contribution: First systematic cross-dataset evaluation of open-set CAN IDS; demonstrates that "
            "few-shot adaptation is viable for real-world deployment. "
            "Limitation: Assumes access to a few labeled samples from the target domain; concept drift over "
            "time is not modeled. Evaluation limited to public datasets without real vehicle validation. "
            "Future work: Investigate continual learning for evolving attacks, test on automotive Ethernet "
            "and zonal architectures, and explore federated few-shot learning across vehicle fleets."
        ),
    },

    # ============================================================
    # DIRECTION 4: Explainable and Deterministic CAN IDS
    # ============================================================
    {
        "id": "explainable_can_ids",
        "title": "Logic Extraction and Non-Negative Matrix Factorization for Explainable CAN Intrusion Detection",
        "abstract": (
            "We propose an interpretable CAN IDS that distills deep model decisions into compact boolean rules "
            "and uses Non-Negative Matrix Factorization (NMF) for lightweight anomaly scoring. The logic "
            "extraction framework achieves >99.97% accuracy with 0.15 microsecond inference latency, making "
            "it suitable for ECU deployment. NMF provides additive, part-based explanations that map directly "
            "to CAN message features. "
            "Contribution: Bridges deep learning and rule-based systems; provides deterministic, auditable "
            "decisions compliant with ISO 21434. "
            "Limitation: Rule extraction struggles with rare attacks and cannot generalize to unseen patterns; "
            "NMF assumes linearity which may not hold for complex attacks. Open-set detection is not addressed. "
            "Future work: Extend to open-set scenarios via rule-based novelty detection, combine with "
            "contrastive learning for better feature separation, and validate on CAN-FD and real vehicles."
        ),
    },

    # ============================================================
    # DIRECTION 5: Synthetic Data Generation for CAN IDS
    # ============================================================
    {
        "id": "synthetic_can_data",
        "title": "Synthetic Attack Generation for Imbalanced CAN Intrusion Detection",
        "abstract": (
            "We address severe class imbalance in CAN datasets by generating synthetic attack samples using "
            "Restricted Boltzmann Machines (RBMs) and Generative Adversarial Networks (GANs). Synthetic data "
            "augmentation improves CANet accuracy from 0.6477 to 0.9725 on minority attack classes. We "
            "evaluate fidelity, diversity, and downstream detection performance. "
            "Contribution: Systematic comparison of generative models for CAN data augmentation; demonstrates "
            "that synthetic data can match real data for rare attack classes. "
            "Limitation: Generated samples may not preserve protocol semantics or physical attack effects; "
            "no validation on real vehicle hardware. GAN training is unstable on small datasets. "
            "Future work: Combine with diffusion models, enforce protocol constraints during generation, "
            "validate synthetic attacks on real CAN buses, and extend to CAN-FD and automotive Ethernet."
        ),
    },

    # ============================================================
    # DIRECTION 6: LLM/VLM-Powered Explainable CAN IDS
    # ============================================================
    {
        "id": "llm_vlm_can_ids",
        "title": "LLM and VLM-Powered Explainable Intrusion Detection for In-Vehicle Networks",
        "abstract": (
            "We fine-tune Large Language Models (LLaMA-3, Phi-3) and Vision-Language Models for CAN intrusion "
            "detection and explanation generation. By converting CAN sequences into textual or image "
            "representations, the model not only classifies attacks but generates human-readable explanations "
            "for its decisions. This addresses the explainability gap in automotive security and supports "
            "ISO 21434 compliance. "
            "Contribution: First application of VLMs to CAN IDS; demonstrates that explanation quality can be "
            "evaluated automatically and correlates with detection accuracy. "
            "Limitation: Full LLMs are not deployable on ECUs; inference cost is prohibitive for real-time "
            "detection. Requires cloud offloading or knowledge distillation. Dataset bias limits generalization. "
            "Future work: Distill LLM explanations into small models, develop CAN-specific prompt engineering, "
            "explore retrieval-augmented generation (RAG) for attack knowledge, and validate on real vehicles."
        ),
    },

    # ============================================================
    # DIRECTION 7: Vision Transformers (ViTs) for CAN Traffic Analysis
    # ============================================================
    {
        "id": "vit_can_ids",
        "title": "Vision Transformers for CAN Traffic Analysis via Image-Based Representations",
        "abstract": (
            "We convert CAN message windows into 2D image representations (recurrence plots, heatmaps) and "
            "apply Vision Transformers (ViT, Swin Transformer) for intrusion detection. IVN-ViT achieves "
            "state-of-the-art accuracy on Car-Hacking and ROAD while providing attention-based interpretability. "
            "We compare against CNN and ResNet baselines. "
            "Contribution: First systematic evaluation of ViTs for CAN IDS; demonstrates that image-based "
            "representations enable transfer learning from computer vision. "
            "Limitation: Image conversion loses temporal precision; ViTs are parameter-heavy and not "
            "ECU-deployable without compression. Open-set detection is unexplored. "
            "Future work: Develop lightweight ViT variants for edge deployment, combine with knowledge "
            "distillation, explore self-supervised pre-training on unlabeled CAN data, and extend to CAN-FD."
        ),
    },

    # ============================================================
    # DIRECTION 8: Multimodal Fusion (CAN + LiDAR/Camera)
    # ============================================================
    {
        "id": "multimodal_can_ids",
        "title": "Multimodal Fusion of CAN and Sensor Data for Robust Automotive Intrusion Detection",
        "abstract": (
            "We propose a multimodal fusion framework that combines CAN bus data with LiDAR and camera data "
            "from public datasets (nuScenes, Waymo) to detect physical and cyber attacks. The hypothesis is "
            "that attacks create inconsistencies between CAN signals and the physical world as perceived by "
            "sensors. We design separate encoders for CAN (GNN/Transformer) and sensors (CNN/ViT) and fuse "
            "their embeddings for classification. "
            "Contribution: First multimodal fusion approach for automotive IDS; demonstrates that sensor data "
            "improves detection of stealthy attacks. "
            "Limitation: Requires synchronized multimodal datasets which are scarce; fusion architecture is "
            "computationally expensive and not ECU-deployable. Real-time constraints are not addressed. "
            "Future work: Develop efficient fusion architectures, explore cross-modal distillation, validate "
            "on real vehicles with hardware-in-the-loop, and extend to V2X communication."
        ),
    },

    # ============================================================
    # DIRECTION 9: GNN + Zero-Shot/Few-Shot Learning with LLMs
    # ============================================================
    {
        "id": "gnn_llm_zeroshot_can",
        "title": "GNN and LLM-Guided Zero-Shot Learning for Open-World CAN Intrusion Detection",
        "abstract": (
            "We combine Graph Neural Networks with LLM-generated semantic descriptions to enable zero-shot "
            "detection of unseen CAN attacks. The LLM generates textual descriptions of attack classes from "
            "ROAD and Car-Hacking datasets, which are used to create semantic prototypes in the GNN embedding "
            "space. For a new attack, the LLM generates its description and the GNN classifies it without "
            "any labeled examples. "
            "Contribution: First zero-shot CAN IDS using LLM-generated semantics; demonstrates that language "
            "models can bridge the gap between known and unknown attacks. "
            "Limitation: LLM descriptions may not capture protocol-level attack semantics; zero-shot accuracy "
            "is lower than few-shot and closed-set. Requires access to an LLM at training time only. "
            "Future work: Improve attack description quality via prompt engineering, combine with few-shot "
            "adaptation, distill LLM knowledge into lightweight models, and validate on CAN-FD and Ethernet."
        ),
    },

    # ============================================================
    # DIRECTION 10: Edge AI and Model Compression for Automotive Security
    # ============================================================
    {
        "id": "edge_ai_can_ids",
        "title": "Edge AI and Model Compression for Real-Time Automotive Intrusion Detection",
        "abstract": (
            "We study model compression techniques (pruning, quantization, knowledge distillation, neural "
            "architecture search) for deploying CAN IDS on automotive-grade hardware (Jetson, ARM Cortex, "
            "FPGA). We benchmark accuracy, latency, memory, and energy across compression methods and "
            "backbones (CNN, LSTM, Transformer, GNN). "
            "Contribution: Comprehensive benchmark of compression techniques for CAN IDS; provides practical "
            "guidelines for ECU deployment. "
            "Limitation: Compression degrades open-set detection and rare attack classes; hardware "
            "validation limited to Jetson, not real ECU. No standardized automotive benchmark exists. "
            "Future work: Develop compression-aware training for open-set IDS, explore hardware-aware NAS, "
            "validate on real ECUs, and standardize automotive IDS benchmarks."
        ),
    },
    {
        "id": "neg_hairclip",
        "title": "HairCLIP: Design Your Hair by Text and Reference Image",
        "abstract": ("We present HairCLIP, unified hair editing via text and reference image conditions using "
            "modulated StyleGAN. Limitation: struggles with extreme poses. Future work: diffusion backbone."),
    },
    {
        "id": "neg_alphafold",
        "title": "AlphaFold Protein Structure Prediction",
        "abstract": ("We predict protein 3D structures from amino acid sequences with attention networks. "
            "Limitation: multi-chain complexes less accurate. Future work: dynamics prediction."),
    },
    {
        "id": "neg_legal_llm",
        "title": "LLM Summarization of Legal Contracts",
        "abstract": ("We fine-tune LLMs to summarize legal contracts with citation grounding. "
            "Limitation: hallucinates clauses on long docs. Future work: retrieval grounding."),
    },
]

print([s["id"] for s in SAMPLES])

['kd_can_ids', 'slm_can_ids', 'openset_fewshot_can', 'explainable_can_ids', 'synthetic_can_data', 'llm_vlm_can_ids', 'vit_can_ids', 'multimodal_can_ids', 'gnn_llm_zeroshot_can', 'edge_ai_can_ids', 'neg_hairclip', 'neg_alphafold', 'neg_legal_llm']


## 2. Extraction prompt v1 (the one pipeline would use)

In [3]:
PROMPT = """You extract structured research info. Return Markdown with exactly these headings.
Thesis: {thesis}
Title: {title}
Abstract: {abstract}

Output:
## Novelty (2 bullets)
## Methodology (3 bullets)
## Explicit Limitations (bullets, quote if stated)
## Future Work (bullets)
## Concept Hubs (exactly 3-5, each MUST match [[Concept - <2-4 word noun>]], reuse existing names verbatim when possible, no other prefix)
## Relevance Score (0-10 + reason; 0-3 off-topic, 4-6 tangential, 7-8 related, 9-10 direct; be strict)
"""
print(PROMPT[:300])

You extract structured research info. Return Markdown with exactly these headings.
Thesis: {thesis}
Title: {title}
Abstract: {abstract}

Output:
## Novelty (2 bullets)
## Methodology (3 bullets)
## Explicit Limitations (bullets, quote if stated)
## Future Work (bullets)
## Concept Hubs (exactly 3-5,


In [4]:
for s in SAMPLES:
    out = chat(PROMPT.format(thesis=THESIS, title=s["title"], abstract=s["abstract"]))
    print(f"\n{'='*80}\n# {s['id']}: {s['title']}\n{'='*80}")
    print(out)


# kd_can_ids: Knowledge Distillation for Ultra-Lightweight Intrusion Detection in Controller Area Networks
## Novelty (2 bullets)
- First systematic study of knowledge distillation for graph-based CAN intrusion detection.
- Demonstrates on-ECU deployment feasibility with <10K parameters, <10 KB memory, and >99% of teacher accuracy.

## Methodology (3 bullets)
- Trains high-capacity teacher models, including Graph Neural Networks and Transformers, on public CAN datasets: Car-Hacking, ROAD, and CAN-FD.
- Distills knowledge into a tiny student model using depthwise separable convolutions and pruned GNN layers.
- Evaluates the student model on ARM Cortex-M4 hardware, reporting inference latency under 1 ms.

## Explicit Limitations (bullets, quote if stated)
- “Distillation is evaluated only on closed-set classification; open-set detection degrades significantly in the student model.”
- “Cross-dataset generalization remains untested.”

## Future Work (bullets)
- Combine distillation with o

## 3. Judge (fill manually)
- Novelty specific? 1-5:
- Limitations vs hallucinations? 1-5:
- Concept Hubs reusable (`[[Concept - X]]`)? 1-5:
- Score calibrated to thesis? 1-5:

If avg <3.5 we tune prompt/temp before building `pipeline/extract.py`. Paste scores back and I lock the prompt.